# Chronos-2 × FEV input-strategy smoke

Runs one **calibration** origin of `epf_np` with `target_only` and `all_dynamic`. This validates GPU inference and FEV scoring only. The resulting numbers are `smoke_only` and must not be used as paper evidence or to change P0 thresholds.

Select **Runtime → Change runtime type → T4 GPU** before running all cells.

In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/covariate-safe-tsfm')
if not REPO.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/FlyMe2star/covariate-safe-tsfm.git', str(REPO)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'chronos-forecasting==2.2.2',
        ('git+https://github.com/autogluon/fev.git@'
         '38007871dcf6dc6b04aed3a54d9cd86678d48d0b'),
    ],
    check=True,
)

SOURCE_ROOT = str(REPO / 'src')
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)
importlib.invalidate_caches()
covsafe = importlib.import_module('covsafe')
print('Repository ready:', REPO)
print('covsafe import:', covsafe.__file__)

In [ ]:
import torch

assert torch.cuda.is_available(), (
    'Select Runtime > Change runtime type > T4 GPU, then restart.'
)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Torch:', torch.__version__)

In [ ]:
import json

from covsafe.chronos2_smoke import (
    EXPECTED_SMOKE_CONFIG_HASH,
    run_chronos2_smoke,
)

print('Frozen smoke config hash:', EXPECTED_SMOKE_CONFIG_HASH)
manifest = run_chronos2_smoke(REPO)
print(json.dumps(manifest, indent=2, ensure_ascii=False, default=str))

In [ ]:
import shutil

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
local_manifest = REPO / 'outputs/smoke/chronos2_epf_np.json'
drive_manifest = (
    Path('/content/drive/MyDrive/covariate-safe-tsfm/private_manifests')
    / 'chronos2_epf_np_smoke.json'
)
drive_manifest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(local_manifest, drive_manifest)
print('Durable manifest:', drive_manifest)

## Return artifact

Send the JSON printed by the third code cell. A valid smoke manifest must report `result_status: smoke_only`, both variants, finite SQL/WQL/MASE/WAPE metrics, `scientific_gate_computed: false`, and `sealed_evaluation_origins_instantiated: false`.